In [10]:
from CRNs import *
import numpy as np
import matplotlib.pyplot as plt
import sympy
import networkx as nx
import random

# --- Example usage with new class structure ---

NR = 1
NL = 5
NL_in = 4
prob = 0.1
species_names, reaction_strings, L, input_indices = generate_random_signaling_network(NR, NL, NL_in, prob, include_reverse=False, include_uncatalyzed=True)
print(input_indices)
print(reaction_strings)

# NR = 1
# NL_vec = [2,2]
# species_names, reaction_strings, L = generate_feedforward_signaling_network(NR, NL_vec)

r_n = ReactionNetwork.from_reaction_strings(
    reaction_strings=reaction_strings,
    L=L,
    seed=42,
    species_names=species_names,
    force_reverse=True
)

# 1. Create the simulator
sim = ReactionNetworkSimulator(r_n)

print("Complexes per class:", r_n.complexes_per_class)
print("Reactions per class:", r_n.reactions_per_class)
print("Extra conservations", r_n.n_species - np.linalg.matrix_rank(r_n.Y @ r_n.A) - r_n.n_cons)

for i, node_assignment in enumerate(r_n.assignments):
    print(f"Linkage group {i+1}: {node_assignment}")

# symbolic ODEs
symbolic_rhs, species, rates = sim.get_symbolic_rhs()
reduced_rhs, remaining_syms, const_syms, rate_syms = sim.get_symbolic_reduced_rhs()



cycles = compute_cycles(r_n)


# Visualize linkage classes
# sz = 3
# fig, axes = plt.subplots(1, len(r_n.linkage_groups), figsize=(sz*len(r_n.linkage_groups), sz))

# if len(r_n.linkage_groups) == 1:
#     pos = nx.spring_layout(r_n.linkage_groups[0])
#     nx.draw(r_n.linkage_groups[0], pos=pos, with_labels=True, ax=axes, node_size=1000, connectionstyle='arc3,rad=0.2', arrows=True)
#     axes.set_title("Linkage class 1")
# else:
#     for i, (G, ax) in enumerate(zip(r_n.linkage_groups, axes)):
#         pos = nx.spring_layout(G)
#         nx.draw(G, pos=pos, with_labels=True, ax=ax, node_size=1000, connectionstyle='arc3,rad=0.2', arrows=True)
#         ax.set_title(f"Linkage class {i+1}")
# plt.tight_layout()
# plt.show()

# 2. Reduce the ODE system using conservation laws
sim.solve_conservation_laws()
print("Remaining species after reduction:", sim.remaining_species)
print("Conservation constants (symbols):", sim.const_syms)
print("Eliminated species solutions:", sim.elimination_solutions)

[1 0 4 3]
['R0+L1 -> R0+L1s', 'R0+L0 -> R0+L0s', 'R0+L4 -> R0+L4s', 'R0+L3 -> R0+L3s', 'L2s+L3 -> L2s+L3s', 'L0 -> L0s', 'L1 -> L1s', 'L2 -> L2s', 'L3 -> L3s', 'L4 -> L4s']
Complexes per class: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Reactions per class: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Extra conservations 0
Linkage group 1: {'L0s': 'L0s', 'L0': 'L0'}
Linkage group 2: {'L1s': 'L1s', 'L1': 'L1'}
Linkage group 3: {'L2s': 'L2s', 'L2': 'L2'}
Linkage group 4: {'L2s+L3': 'L2s+L3', 'L2s+L3s': 'L2s+L3s'}
Linkage group 5: {'L3s': 'L3s', 'L3': 'L3'}
Linkage group 6: {'L4s': 'L4s', 'L4': 'L4'}
Linkage group 7: {'R0+L0': 'R0+L0', 'R0+L0s': 'R0+L0s'}
Linkage group 8: {'R0+L1': 'R0+L1', 'R0+L1s': 'R0+L1s'}
Linkage group 9: {'R0+L3s': 'R0+L3s', 'R0+L3': 'R0+L3'}
Linkage group 10: {'R0+L4s': 'R0+L4s', 'R0+L4': 'R0+L4'}
Remaining species after reduction: ['L0s', 'L1s', 'L2s', 'L3s', 'L4s']
Conservation constants (symbols): [const0, const1, const2, const3, const4, const5]
Eliminated species solutions: {'L0': -L0s 

In [11]:
interaction_matrix = get_interaction_matrix(r_n)
print(interaction_matrix)
print(np.sum(interaction_matrix) / (2*len(r_n.get_conservation_groups())**2))


[[0. 2. 2. 0. 2. 2.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 2. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]
0.1388888888888889


In [46]:
E_range = 1
B_range = 1
F_range = 1
C0 = 1
beta = 1
reac_rates = generate_thermodynamic_rates(r_n, C0, beta, E_range, B_range, F_range) 
r_n.update_rates(reac_rates)

t_span = (0, 10000)
num_points = 2000  
r_tol = 1e-6    
a_tol = 1e-6


M_0t1, M_1t0, M_0b1 = count_conservation_group_changes(r_n)


sim = ReactionNetworkSimulator(r_n)
sim.solve_conservation_laws()
reduced_rhs, remaining_syms, const_syms, rate_syms = sim.get_symbolic_reduced_rhs()
dR_dC, dR_dC_func, dR_dl, dR_dl_func, dR_dk, dR_dk_func, remaining_syms = sim.get_first_order_derivatives()

# E_range = 1
# B_range = 1
# F_range = 5
# rates = generate_thermodynamic_rates(r_n, C0, beta, E_range, B_range, F_range) 
# r_n.update_rates(rates)


rates = np.array([r_n.reactions[r_idx][2] for r_idx in range(len(r_n.reactions))])
flexible_reduced_ode_rhs = sim.make_reduced_rhs_with_conservation_flexible()


l0 = np.exp(np.random.uniform(np.log(0.0001), np.log(1000.0), size=L.shape[0]))  # Sample uniformly in log space between 0.1 and 1000

C_full = generate_positive_initial_concentrations_nnls(L, l0)
_, C_reduced_init = sim.get_const_and_reduced_init(C_full)
sim.make_reduced_rhs_with_conservation(l0)


sol_reduced, C_reduced_final = sim.integrate(
    lambda C: flexible_reduced_ode_rhs(C, l0), C_reduced_init, t_span=t_span,
    num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol, vectorized=True
)

C_full = sim.recover_eliminated_species(l0, C_reduced_final)
dC_dl = sim.dC_dl_func(C_reduced_final, l0, rates, dR_dC_func, dR_dl_func)
dC_dl_full = sim.compute_dC_dk_full(dC_dl, l_bool = True)
signs = np.sign(np.round(dC_dl_full, decimals=12)).tolist()

print(C_full)


[8.22279125e+02 1.33414979e-04 3.95243749e-05 8.54181095e-04
 9.83415673e-04 2.36337506e+00 1.67558751e+00 6.67861656e+01
 3.88040274e+01 1.23233104e-02 1.25697183e-02]


In [47]:
import sympy as sp
adj_dR_dC = dR_dC.adjugate()
print("Adjugate matrix:")
print(adj_dR_dC)

Adjugate matrix:
Matrix([[(L1s*k_17 - k_16*(-L1s + const2))*(-L3s*k_11 + k_10*(-L3s + const4))*((-L2s*k_19 + k_18*(-L2s + const3))*(L3s*k_13 - k_12*(-L3s + const4)) + (L1s*k_8 + L1s*k_9 + L3s*k_18 + L3s*k_19 + k_30 + k_31)*(L2s*k_14 + L2s*k_15 + L3s*k_20 + L3s*k_21 + k_34 + k_35) + (L0s*k_4 + L0s*k_5 + L1s*k_10 + L1s*k_11 + L2s*k_12 + L2s*k_13 + const0*k_0 + const0*k_1 + k_32 + k_33)*(L1s*k_8 + L1s*k_9 + L2s*k_14 + L2s*k_15 + L3s*k_18 + L3s*k_19 + L3s*k_20 + L3s*k_21 + k_30 + k_31 + k_34 + k_35)) + (L1s*k_17 - k_16*(-L1s + const2))*((-L3s*k_13 + k_12*(-L3s + const4))*((-L2s*k_19 + k_18*(-L2s + const3))*(-L3s*k_11 + k_10*(-L3s + const4)) + (-L2s*k_9 + k_8*(-L2s + const3))*(-L1s*k_8 - L1s*k_9 - L3s*k_18 - L3s*k_19 - k_30 - k_31)) + ((-L2s*k_9 + k_8*(-L2s + const3))*(-L3s*k_13 + k_12*(-L3s + const4)) + (-L3s*k_11 + k_10*(-L3s + const4))*(-L0s*k_4 - L0s*k_5 - L1s*k_10 - L1s*k_11 - L2s*k_12 - L2s*k_13 - const0*k_0 - const0*k_1 - k_32 - k_33))*(-L0s*k_4 - L0s*k_5 - L1s*k_10 - L1s*k_11 - L2s*

In [8]:
print(adj_dR_dC[0,0])

-4*E**3*k_2*k_5*k_9 - 4*E**3*k_2*k_7*k_9 - 4*E**3*k_4*k_7*k_9 - 2*E**2*k_2*k_6*k_9 - 2*E**2*k_3*k_4*k_9 - E**2*k_3*k_5*k_9 - E**2*k_3*k_7*k_9 - 2*E**2*k_4*k_6*k_9 - E**2*k_5*k_6*k_9 + E*F*k_3*k_5*k_9 + E*F*k_3*k_7*k_9 + E*F*k_5*k_6*k_9


In [ ]:
r_tol = 1e-12
a_tol = 1e-12
start_time = time.time()
for _ in range(20):
    sol_reduced, C_reduced_final = sim.integrate(
    lambda C: flexible_reduced_ode_rhs(C, l0), C_reduced_init, t_span=t_span,
    num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol
)
print(time.time() - start_time)

start_time = time.time()
for _ in range(20):
    sol_reduced, C_reduced_final = sim.integrate(
    lambda C: flexible_reduced_ode_rhs(C, l0), C_reduced_init, t_span=t_span,
    num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol
)
print(time.time() - start_time)

0.21371221542358398
0.20806598663330078


In [5]:
import numpy as np
from scipy.interpolate import PPoly

# Create a grid of C values and evaluate the polynomial function
C_min = np.min(C_reduced_final) - 0.1
C_max = np.max(C_reduced_final) + 0.1
C_grid = np.linspace(C_min, C_max, 1000)

# Evaluate the polynomial function at grid points
y_values = np.array([flexible_reduced_ode_rhs(C, l0) for C in C_grid])

# Create PPoly object and find roots
ppoly = PPoly.from_spline_data(C_grid, y_values)
roots = ppoly.solve(0)  # Find where polynomial = 0

print(f"Found roots: {roots}")

TypeError: 'numpy.float64' object is not iterable

In [20]:
# Get baseline solution

h = 0.0001
C_full_baseline = generate_positive_initial_concentrations_nnls(L, l0)
_, C_reduced_init_baseline = sim.get_const_and_reduced_init(C_full_baseline)
sim.make_reduced_rhs_with_conservation(l0)

sol_baseline, C_reduced_final_baseline = sim.integrate(
    lambda C: flexible_reduced_ode_rhs(C, l0), C_reduced_init_baseline, 
    t_span=t_span, num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol
)

C_full_baseline = sim.recover_eliminated_species(l0, C_reduced_final_baseline)

# Finite difference approximation
dC_dl0_finite_diff = np.zeros_like(dC_dl)
dC_dl_full_finite_diff = np.zeros_like(dC_dl_full)

for i in range(len(l0)):
    # Perturb l0[i] by h
    l0_perturbed = l0.copy()
    l0_perturbed[i] += h
    
    # Get solution with perturbed l0
    C_full_perturbed = generate_positive_initial_concentrations_nnls(L, l0_perturbed)
    _, C_reduced_init_perturbed = sim.get_const_and_reduced_init(C_full_perturbed)
    sim.make_reduced_rhs_with_conservation(l0_perturbed)
    
    sol_perturbed, C_reduced_final_perturbed = sim.integrate(
        lambda C: flexible_reduced_ode_rhs(C, l0_perturbed), C_reduced_init_perturbed, 
        t_span=t_span, num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol
    )
    C_full = sim.recover_eliminated_species(l0_perturbed, C_reduced_final_perturbed)

    # Finite difference approximation: (C_perturbed - C_baseline) / h
    dC_dl0_finite_diff[:, i] = (C_reduced_final_perturbed - C_reduced_final_baseline) / h
    dC_dl_full_finite_diff[:, i] = (C_full - C_full_baseline) / h

In [22]:
print(dC_dl0_finite_diff.round(3))
print(dC_dl_full_finite_diff.round(3))


[[ 0.163  0.     0.   ]
 [ 0.147 -0.    -0.   ]
 [-0.     0.277 -0.   ]
 [ 0.     0.53   0.   ]
 [ 0.     0.     0.361]
 [ 0.     0.     0.405]
 [ 0.     0.     0.148]]
[[ 0.69   0.     0.   ]
 [ 0.163  0.     0.   ]
 [ 0.147 -0.    -0.   ]
 [ 0.     0.193  0.   ]
 [-0.     0.277 -0.   ]
 [ 0.     0.53   0.   ]
 [-0.    -0.     0.086]
 [ 0.     0.     0.361]
 [ 0.     0.     0.405]
 [ 0.     0.     0.148]]


In [21]:
print(dC_dl.round(3))
print(dC_dl_full.round(3))

[[ 0.163  0.     0.   ]
 [ 0.147 -0.    -0.   ]
 [-0.     0.277 -0.   ]
 [ 0.     0.53   0.   ]
 [ 0.    -0.     0.361]
 [ 0.    -0.     0.405]
 [ 0.    -0.     0.148]]
[[ 0.69  -0.     0.   ]
 [ 0.163  0.     0.   ]
 [ 0.147 -0.    -0.   ]
 [-0.     0.193  0.   ]
 [-0.     0.277 -0.   ]
 [ 0.     0.53   0.   ]
 [-0.     0.     0.086]
 [ 0.    -0.     0.361]
 [ 0.    -0.     0.405]
 [ 0.    -0.     0.148]]


In [22]:
1-0.855014

0.14498599999999995

array([[-0.855014, -0.      , -0.      ],
       [ 0.782694,  0.      ,  0.      ],
       [ 0.07232 ,  0.      ,  0.      ],
       [ 0.      , -0.339337, -0.      ],
       [-0.      ,  0.262498,  0.      ],
       [-0.      ,  0.076839,  0.      ],
       [-0.      ,  0.      , -0.887266],
       [ 0.      , -0.      ,  0.430036],
       [-0.      ,  0.      ,  0.096773],
       [ 0.      , -0.      ,  0.360457]])

In [180]:
from scipy.optimize import root
import numpy as np


# Define the function to find the root of (steady state condition)
def steady_state_condition(C_squared, l0, flexible_reduced_ode_rhs):
    """
    Function that should be zero at steady state.
    Uses square transform to ensure positive solutions.
    C_squared represents the squared concentrations.
    """
    # Transform back from squared to actual concentrations
    C = C_squared**2
    
    # Evaluate the RHS at the actual concentrations
    rhs = flexible_reduced_ode_rhs(C, l0)
    
    return rhs

# Helper function for the original Jacobian (without square transform)
def steady_state_jacobian_original(C, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms):
    """
    Original Jacobian without square transform.
    """
    # Create arguments in the correct order: remaining_syms + const_syms + rate_syms
    args = list(C) + list(l0) + list(rates)
    
    # Call dR_dC_func with the arguments in the expected order
    return dR_dC_func(*args)

# Define the Jacobian function with square transform
def steady_state_jacobian(C_squared, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms):
    """
    Analytical Jacobian of the steady state condition with square transform.
    Uses chain rule: dR/d(C²) = dR/dC * dC/d(C²) = dR/dC * (1/(2*C))
    """
    # Transform back from squared to actual concentrations
    C = C_squared**2
    
    # Get the Jacobian w.r.t. actual concentrations
    dR_dC = steady_state_jacobian_original(C, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms)

    # Create diagonal matrix for the chain rule factor
    chain_rule_factor = np.diag(2.0 * C_squared)
    
    # Apply chain rule
    dR_dC_squared = dR_dC @ chain_rule_factor
    
    return dR_dC_squared


# Find the root using the square transform
result = root(
    fun=lambda C_squared: steady_state_condition(C_squared, l0, flexible_reduced_ode_rhs),
    x0=C_reduced_init_squared,  # Use the squared initial condition as starting point
    #jac=lambda C_squared: steady_state_jacobian(C_squared, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms),
    method='hybr',  # Powell's hybrid method
    options={'xtol': 1e-12}
)

print("Steady state found successfully!")
#print(f"Number of function evaluations: {result.nfev}")
print(f"Final function value norm: {np.linalg.norm(result.fun)}")

# Transform back from squared space to actual concentrations
C_reduced_final_root = result.x**2
print(C_reduced_final)
print(np.linalg.norm(flexible_reduced_ode_rhs(C_reduced_final, l0)))

t_span = (0, 10000)
num_points = 200009  
r_tol = 1e-12    
a_tol = 1e-12

sol_reduced, C_reduced_final_int = sim.integrate(
    lambda C: flexible_reduced_ode_rhs(C, l0), C_reduced_init, t_span=t_span,
    num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol
)
#C_reduced_steady = C_reduced_final
print(C_reduced_final_int)
print(np.linalg.norm(flexible_reduced_ode_rhs(C_reduced_final_int, l0)))

print(np.linalg.norm(C_reduced_final_int-C_reduced_final_root))


Steady state found successfully!
Final function value norm: 3.278292799127577
[6.16201097e-05 3.30394539e-02 1.52048657e+01 1.18092874e+01
 2.44674602e+00 4.52593766e-01 9.84934161e-01]
0.0006751258533613746
[1.54967421e-02 8.55989790e-03 1.52048657e+01 1.18092874e+01
 5.87662321e-01 8.39544431e-01 1.82701586e+00]
1.7288782077954873e-13
1.8869204308891097


In [128]:
from scipy.optimize import minimize
import numpy as np

# Define the objective function (squared norm of RHS)
def objective_function(C_squared, l0, flexible_reduced_ode_rhs):
    """
    Objective function: ||RHS(C)||²
    Uses square transform to ensure positive solutions.
    """
    # Transform back from squared to actual concentrations
    C = C_squared**2
    
    # Evaluate the RHS at the actual concentrations
    rhs = flexible_reduced_ode_rhs(C, l0)
    
    # Return the squared norm
    return 0.5 * np.sum(rhs**2)

# Define the gradient of the objective function
def objective_gradient(C_squared, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms, flexible_reduced_ode_rhs):
    """
    Gradient of ||RHS(C)||² with respect to C_squared
    """
    # Transform back from squared to actual concentrations
    C = C_squared**2
    
    # Get the RHS and Jacobian
    rhs = flexible_reduced_ode_rhs(C, l0)
    jac = steady_state_jacobian_original(C, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms)
    
    # Apply chain rule: d/d(C²) = d/dC * dC/d(C²)
    # dC/d(C²) = 1/(2*C_squared) = 1/(2*sqrt(C))
    chain_rule_factor = np.diag(1.0 / (2.0 * C_squared))
    
    # Gradient: J^T * RHS * chain_rule_factor
    gradient = jac.T @ rhs @ chain_rule_factor
    
    return gradient

# Transform initial condition to squared space
C_reduced_init_squared = np.sqrt(np.abs(C_reduced_init))

# Use scipy minimize with conjugate gradient
result = minimize(
    fun=lambda C_squared: objective_function(C_squared, l0, flexible_reduced_ode_rhs),
    x0=C_reduced_final_int,
    #jac=lambda C_squared: objective_gradient(C_squared, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms, flexible_reduced_ode_rhs),
    method='CG',  # Conjugate gradient
    options={
        'gtol': 1e-8,
        'maxiter': 1000,
        'disp': True
    }
)

C_reduced_final = result.x**2
print(C_reduced_final)
print(np.linalg.norm(flexible_reduced_ode_rhs(C_reduced_final, l0)))


         Current function value: 0.051489
         Iterations: 210
         Function evaluations: 4795
         Gradient evaluations: 598
[1.37783742e+02 2.35076016e+00 6.28344867e-03 6.89457206e-05
 1.59675248e-03 4.45083042e-09 5.93956573e+02]
0.3209009722991462


In [129]:
# Modified root finding with diagonal preconditioning
def preconditioned_steady_state_condition(C_squared, l0, flexible_reduced_ode_rhs):
    """Preconditioned steady state condition"""
    f_val = steady_state_condition(C_squared, l0, flexible_reduced_ode_rhs)
    
    # Diagonal preconditioning
    scale_factors = np.maximum(np.abs(C_squared), 1e-8)
    return f_val / scale_factors

def preconditioned_jacobian(C_squared, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms):
    """Preconditioned Jacobian"""
    jac = steady_state_jacobian(C_squared, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms)
    scale_factors = np.maximum(np.abs(C_squared), 1e-8)
    
    # Apply diagonal preconditioning: D^(-1) * J * D
    D_inv = np.diag(1.0 / scale_factors)
    D = np.diag(scale_factors)
    return D_inv @ jac @ D

# Use in root finding
result = root(
    fun=lambda C_squared: preconditioned_steady_state_condition(C_squared, l0, flexible_reduced_ode_rhs),
    x0=C_reduced_final_int,
    jac=lambda C_squared: preconditioned_jacobian(C_squared, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms),
    method='hybr',
    options={'xtol': 1e-12}
)

print("Steady state found successfully!")
print(f"Number of function evaluations: {result.nfev}")
print(f"Final function value norm: {np.linalg.norm(result.fun)}")

# Transform back from squared space to actual concentrations
C_reduced_final = result.x**2
print(C_reduced_final)

Steady state found successfully!
Number of function evaluations: 199
Final function value norm: 363311.97631146
[   5.99505409  168.96579937   53.56010279 1888.11951848   61.91377806
  110.66125049  734.72584093]


In [62]:
start_time = time.time()
for _ in range(10):
    result = root(
        fun=lambda C_squared: steady_state_condition(C_squared, l0, flexible_reduced_ode_rhs),
        x0=C_reduced_init_squared,  # Use the squared initial condition as starting point
        jac=lambda C_squared: steady_state_jacobian(C_squared, l0, dR_dC_func, rates, remaining_syms, const_syms, rate_syms),
        method='hybr',  # Powell's hybrid method
        options={'xtol': 1e-8}
    )
print(time.time() - start_time)


start_time = time.time()
for _ in range(10):
    sol_reduced, C_reduced_final = sim.integrate(
    lambda C: flexible_reduced_ode_rhs(C, l0), C_reduced_init, t_span=t_span,
    num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol
)
print(time.time() - start_time)

0.013604879379272461
0.28154897689819336


In [19]:
steady_state_condition(C_reduced_init_squared, l0, flexible_reduced_ode_rhs)

array([-4.64598222e-06,  4.63121625e-06,  9.54299715e+02,  2.41537103e+03,
        1.71328339e-02, -1.39197873e-07, -1.71378193e-02])

In [7]:
print("Length of l0:", len(l0))
print("Length of const_syms:", len(const_syms))
print("l0:", l0)
print("const_syms:", const_syms)

Length of l0: 3
Length of const_syms: 3
l0: [5.80968245e-04 6.32922935e+00 1.37532637e-03]
const_syms: [const0, const1, const2]


In [ ]:
from scipy.optimize import root

# # Replace the integration with constrained optimization
# def objective_function(C_reduced):
#     """Objective function - sum of squared ODE RHS (we want this to be zero)"""
#     rhs = flexible_reduced_ode_rhs(C_reduced, l0)
#     return np.sum(rhs**2)

# # Use constrained optimization to find steady state with positive concentrations
# # Set bounds to ensure all concentrations are positive
# bounds = [(1e-12, None)] * len(C_reduced_init)  # Small positive lower bound

# result = minimize(objective_function, C_reduced_init, method='L-BFGS-B',
#                   bounds=bounds, options={'maxiter': 10000, 'ftol': 1e-16})

# Transform to ensure positivity: C = exp(x)
# def objective_function_transformed(x):
#     """Objective function with log transformation to ensure positive concentrations"""
#     C_reduced = np.exp(x)  # This ensures C > 0
#     return flexible_reduced_ode_rhs(C_reduced, l0)

# # Transform initial guess to log space
# x_init = np.log(C_reduced_init)
# result = root(objective_function_transformed, x_init, method='hybr',
#                 options={'xtol': 1e-14})
# C_reduced_final = np.exp(result.x)



# def objective_function_transformed(x):
#     C_reduced = x**2  # This ensures C >= 0
#     return flexible_reduced_ode_rhs(C_reduced, l0)

# # Transform initial guess to square space
# x_init = np.sqrt(C_reduced_init)
# result = root(objective_function_transformed, x_init, method='anderson',
#                 options={'xtol': 1e-14, 'ftol': 1e-14})
# C_reduced_final = result.x**2


# print(flexible_reduced_ode_rhs(C_reduced_final, l0))


# def objective_function_minimization(x):
#     C_reduced = x**2  # This ensures C >= 0
#     rhs_vector = flexible_reduced_ode_rhs(C_reduced, l0)
#     # Return the sum of squares of the RHS vector components
#     return np.sum(rhs_vector**2)

# # Transform initial guess to square space
# x_init = np.sqrt(C_reduced_init)

# # Use scipy.optimize.minimize instead of root
# from scipy.optimize import minimize
# result = minimize(objective_function_minimization, x_init, method='Nelder-Mead',
#                   options={'ftol': 1e-16, 'gtol': 1e-18})

# C_reduced_final = result.x**2

#print(flexible_reduced_ode_rhs(C_reduced_final, l0))

# def objective_function_transformed(x):
#     C_reduced = np.log(1 + np.exp(x))  # Softplus: log(1 + exp(x))
#     return flexible_reduced_ode_rhs(C_reduced, l0)

# # Transform initial guess to square space
# x_init = np.log(np.exp(C_reduced_init) - 1)
# # result = root(objective_function_transformed, x_init, method='hybr',
# #                 options={'xtol': 1e-14})
# result = root(objective_function_transformed, x_init, method='lm',
#                 options={'xtol': 1e-14})
# C_reduced_final = np.log(1 + np.exp(result.x))

# start_time = time.time()
# def objective_function(C):
#     return np.sum(flexible_reduced_ode_rhs(C, l0)**2)

# # Use constrained optimization with positivity bounds
# result = minimize(
#     objective_function,
#     C_reduced_init,
#     method='trust-constr',
#     bounds=[(1e-12, None)] * len(C_reduced_init)
# )

# end_time = time.time()
# print(f"Time taken: {end_time - start_time} seconds")

# result = root(objective_function_transformed, x_init, method='krylov',
#                 options={'xtol': 1e-14})

# start_time = time.time()

# for _ in range(20):
#     # Use root finding on transformed variables
#     result = root(objective_function_transformed, x_init, method='lm',
#                 options={'xtol': 1e-12, 'ftol': 1e-12, 'maxiter': 10000})

# end_time = time.time()
# print(f"Time taken: {end_time - start_time} seconds")



print(C_reduced_final)
print(f"Root finding converged in {result.nfev} function evaluations")

t_span = (0, 10000)
num_points = 2000  
r_tol = 1e-14    
a_tol = 1e-14

start_time = time.time()
for _ in range(20):
# Fallback to integration if root finding fails
    sol_reduced, C_reduced_final = sim.integrate(
        lambda C: flexible_reduced_ode_rhs(C, l0), C_reduced_init, t_span=t_span,
        num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol
    )
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(C_reduced_final)
#print(flexible_reduced_ode_rhs(C_reduced_final, l0))
print(np.sum(flexible_reduced_ode_rhs(C_reduced_final, l0)**2))

def objective_function_transformed(x):
    C_reduced = x**2  # This ensures C >= 0
    return flexible_reduced_ode_rhs(C_reduced, l0)

# Transform initial guess to square space
x_init = np.sqrt(C_reduced_final)
result = root(objective_function_transformed, x_init, method='hybr',
                options={'xtol': 1e-16})
C_reduced_final = result.x**2


print(C_reduced_final)
print(np.sum(flexible_reduced_ode_rhs(C_reduced_final, l0)**2))
